# 02 - Make it intelligent: cash-shortfall ML model + Gen AI enrichment
Runs as task 2 of the `smb-cashflow-copilot` job, right after the Lakeflow ingest pipeline.
1. **ML** - train a gradient-boosted classifier that predicts *P(cash shortfall in next 60 days)* from the gold
   feature table, track it in MLflow, register it in Unity Catalog with alias `champion`.
2. **Score** every SMB -> `account_cashflow_health` (the serving table that is synced to Lakebase).
3. **Gen AI** - embed agent call notes (`databricks-gte-large-en`) for semantic search in Lakebase (pgvector) and
   generate a "next best action" briefing for every High-risk SMB with Claude via `ai_query`.

In [ ]:
%pip install -q "mlflow>=2.16" scikit-learn
%restart_python

In [ ]:
dbutils.widgets.text("catalog", "serverless_stable_0dcqtv_catalog")
dbutils.widgets.text("schema", "smb_cashflow")
dbutils.widgets.text("llm_endpoint", "databricks-claude-sonnet-5-5")
CAT, SCH = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
LLM = dbutils.widgets.get("llm_endpoint")
FQ = f"{CAT}.{SCH}"
spark.sql(f"USE {FQ}")
print("target schema:", FQ, "| llm:", LLM)

target schema: serverless_stable_0dcqtv_catalog.smb_cashflow | llm: databricks-claude-sonnet-5-5


## 1. Training set = gold features joined to historical outcomes

In [ ]:
FEATURES = ["current_balance", "overdue_ar", "overdue_ar_count", "open_ar", "max_days_overdue",
            "ap_due_30", "overdue_bills", "monthly_inflow", "monthly_outflow", "net_monthly",
            "runway_days", "tenure_days"]
df = spark.sql(f"""
  SELECT f.account_id, {', '.join('f.' + c for c in FEATURES)}, o.cash_shortfall_60d AS label
  FROM gold_account_features f JOIN silver_account_outcomes o USING (account_id)""").toPandas()
df[FEATURES] = df[FEATURES].astype(float)
print(df.shape, "| positive rate:", round(df.label.mean(), 3))
df.describe().T[["mean", "min", "max"]].round(1)

(1500, 14) | positive rate: 0.231

                     mean      min       max
current_balance   79707.4  -5856.0  219989.0
overdue_ar        20154.8      0.0  148477.7
overdue_ar_count      2.5      0.0      16.0
open_ar           42076.0      0.0  135607.2
max_days_overdue     76.3      0.0     167.0
ap_due_30         17007.6      0.0   60776.2
overdue_bills         1.3      0.0       7.0
monthly_inflow    51948.6   7240.9  112138.0
monthly_outflow   41089.5   9050.2   88674.5
net_monthly       10859.1 -50804.0   70741.2
runway_days         294.5      0.0     365.0
tenure_days         918.2     63.0    1802.0
label                 0.2      0.0       1.0

In [ ]:
import mlflow
import numpy as np
from mlflow.models import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.model_selection import train_test_split

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/smb_cashflow_shortfall_model")
MODEL_NAME = f"{FQ}.cash_shortfall_model"

X, y = df[FEATURES], df["label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
params = dict(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.9, random_state=42)

with mlflow.start_run(run_name="gbt_cash_shortfall") as run:
    model = GradientBoostingClassifier(**params).fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    # baseline = the old rule (runway < 45 days) that agents use today
    rule = (Xte["runway_days"] < 45).astype(int)
    metrics = {"test_auc": roc_auc_score(yte, p), "test_avg_precision": average_precision_score(yte, p),
               "baseline_rule_auc": roc_auc_score(yte, rule)}
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, name="model", registered_model_name=MODEL_NAME,
                             pyfunc_predict_fn="predict_proba",
                             signature=infer_signature(Xtr, model.predict_proba(Xtr)),
                             input_example=Xtr.head(3))
    print("run_id:", run.info.run_id)
    print({k: round(v, 4) for k, v in metrics.items()})
    print(classification_report(yte, (p >= 0.5).astype(int), digits=3))

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
🔗 View Logged Model at: https://<workspace-host>/ml/experiments/515311966395607/models/m-58bd40cb2cde4c79aab16da78f6d74dd?o=<workspace-id>
Registered model 'serverless_stable_0dcqtv_catalog.smb_cashflow.cash_shortfall_model' already exists. Creating a new version of this model...

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]
🔗 Created version '3' of model 'serverless_stable_0dcqtv_catalog.smb_cashflow.cash_shortfall_model': https://<workspace-host>/explore/data/models/serverless_stable_0dcqtv_catalog/smb_cashflow/cash_shortfall_model/version/3?o=<workspace-id>

run_id: 8848e0419b1548edafb8865ebf8c3105
{'test_auc': 0.7884, 'test_avg_precision': 0.6071, 'baseline_rule_auc': 0.6892}
              precisi

In [ ]:
from mlflow import MlflowClient
client = MlflowClient()
latest = max(client.search_model_versions(f"name='{MODEL_NAME}'"), key=lambda v: int(v.version))
client.set_registered_model_alias(MODEL_NAME, "champion", latest.version)
print(f"registered {MODEL_NAME} v{latest.version} -> alias @champion")
imp = sorted(zip(FEATURES, model.feature_importances_), key=lambda t: -t[1])
for f, w in imp:
    print(f"{f:<18} {w:.3f}")

registered serverless_stable_0dcqtv_catalog.smb_cashflow.cash_shortfall_model v3 -> alias @champion
current_balance    0.549
monthly_inflow     0.101
overdue_ar         0.072
net_monthly        0.051
tenure_days        0.049
open_ar            0.044
ap_due_30          0.041
monthly_outflow    0.036
max_days_overdue   0.030
overdue_ar_count   0.013
runway_days        0.009
overdue_bills      0.005


## 2. Score every SMB with the `@champion` model -> `account_cashflow_health`

In [ ]:
import pandas as pd
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
gold = spark.table("gold_account_features").toPandas()
gold[FEATURES] = gold[FEATURES].astype(float)
gold["shortfall_prob"] = np.round(np.asarray(champion.predict(gold[FEATURES]))[:, 1], 4)  # P(class=1)


def tier(r):
    if r.shortfall_prob >= 0.55 or r.runway_days < 30:
        return "High"
    if r.shortfall_prob >= 0.25 or r.rule_risk_tier != "Healthy":
        return "Watch"
    return "Healthy"


def driver(r):
    if r.overdue_ar > max(r.current_balance, 0) and r.overdue_ar > 0:
        return "Overdue receivables exceed cash on hand"
    if r.net_monthly < 0:
        return "Negative monthly cash flow"
    if r.ap_due_30 > max(r.current_balance, 0):
        return "Bills due in 30 days exceed cash"
    return "Stable"


gold["risk_tier"] = gold.apply(tier, axis=1)
gold["primary_driver"] = gold.apply(driver, axis=1)
gold["model_version"] = int(latest.version)
gold["scored_at"] = pd.Timestamp.utcnow().tz_localize(None)

(spark.createDataFrame(gold).write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("account_cashflow_health"))
spark.sql("ALTER TABLE account_cashflow_health ALTER COLUMN account_id SET NOT NULL")
spark.sql("""ALTER TABLE account_cashflow_health SET TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
try:
    spark.sql("ALTER TABLE account_cashflow_health ADD CONSTRAINT pk_health PRIMARY KEY (account_id)")
except Exception as e:
    print("PK exists:", str(e)[:80])
spark.sql("""COMMENT ON TABLE account_cashflow_health IS
  'Serving table: gold features + ML shortfall probability (cash_shortfall_model@champion), final risk tier and primary driver. Synced to Lakebase for the app.'""")
display(spark.sql("""SELECT risk_tier, count(*) accounts, round(avg(shortfall_prob),3) avg_prob,
                            round(sum(overdue_ar)) overdue_ar FROM account_cashflow_health GROUP BY 1 ORDER BY 1"""))

risk_tier | accounts | avg_prob | overdue_ar
Healthy | 963 | 0.061 | 7800179.0
High | 304 | 0.718 | 14895209.0
Watch | 233 | 0.276 | 7536841.0

In [ ]:
display(spark.sql("""SELECT account_id, business_name, agent_name, current_balance, overdue_ar, runway_days,
                            rule_risk_tier, shortfall_prob, risk_tier, primary_driver
                     FROM account_cashflow_health WHERE account_id = 'SMB-0112'"""))

account_id | business_name | agent_name | current_balance | overdue_ar | runway_days | rule_risk_tier | shortfall_prob | risk_tier | primary_driver
SMB-0112 | Granite Goods | Priya Nair | 21500.0 | 89040.0 | 39.0 | High | 0.7447 | High | Overdue receivables exceed cash on hand

## 3. Gen AI: embed agent notes (semantic search) + Claude next-best-action briefings

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE account_notes_embedded
COMMENT 'Agent call notes + 1024-d gte-large-en embeddings (ai_query). Synced to Lakebase; queried with pgvector.'
TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
SELECT note_id, account_id, agent_id, created_date, note_text,
       ai_query('databricks-gte-large-en', note_text) AS embedding
FROM silver_account_notes""")
spark.sql("ALTER TABLE account_notes_embedded ALTER COLUMN note_id SET NOT NULL")
try:
    spark.sql("ALTER TABLE account_notes_embedded ADD CONSTRAINT pk_notes PRIMARY KEY (note_id)")
except Exception as e:
    print("PK exists:", str(e)[:80])
display(spark.sql("SELECT count(*) notes, min(size(embedding)) dim FROM account_notes_embedded"))

notes | dim
3063 | 1024

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE copilot_briefings
COMMENT 'Claude-generated next-best-action briefing for each High-risk SMB (batch ai_query, grounded only in gold facts)'
TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
SELECT account_id, business_name, agent_id, risk_tier, shortfall_prob,
       ai_query('{LLM}',
         concat('You are a cash-flow copilot for a small-business Customer Success Agent. Using ONLY these facts, ',
                'write a 2-sentence briefing (max 70 words): the cash risk in plain words, then the single best next action. Facts: ',
                'business=', business_name, '; industry=', industry, '; bank balance=$', format_number(current_balance,0),
                '; overdue receivables=$', format_number(overdue_ar,0), ' across ', overdue_ar_count, ' invoices',
                '; bills due in 30 days=$', format_number(ap_due_30,0), '; net monthly cash flow=$', format_number(net_monthly,0),
                '; runway days=', runway_days, '; ML 60-day shortfall probability=', round(shortfall_prob*100), '%',
                '; primary driver=', primary_driver),
         modelParameters => named_struct('max_tokens', 1000)) AS briefing,
       current_timestamp() AS generated_at
FROM account_cashflow_health WHERE risk_tier = 'High'""")
spark.sql("ALTER TABLE copilot_briefings ALTER COLUMN account_id SET NOT NULL")
try:
    spark.sql("ALTER TABLE copilot_briefings ADD CONSTRAINT pk_brief PRIMARY KEY (account_id)")
except Exception as e:
    print("PK exists:", str(e)[:80])
for r in spark.sql("""SELECT account_id, business_name, briefing FROM copilot_briefings
                      ORDER BY account_id = 'SMB-0112' DESC, shortfall_prob DESC LIMIT 4""").collect():
    print(f"[{r.account_id}] {r.business_name}\n  {r.briefing}\n")
print("briefings:", spark.table("copilot_briefings").count())

[SMB-0112] Granite Goods
  Granite Goods is burning $16,500 a month with only $21,500 in the bank (about 39 days of runway), and a 74% chance of a 60-day cash shortfall, mainly because $89,040 across 8 overdue invoices is stuck uncollected, which is more than the cash on hand. The best next action is to start collection calls today on the largest overdue invoices, offering short payment plans if needed, so cash arrives before the $13,758 in bills come due.

[SMB-1092] Lantern Marketing
  Lantern Marketing has only about 7 days of cash ($4,025) while losing $16,584 a month and facing $16,046 in bills within 30 days, and the model puts the chance of a 60-day shortfall at 97%, mainly because $70,607 in overdue receivables far exceeds cash on hand. The best next action is to immediately contact the clients behind the 11 overdue invoices, starting with the largest, to secure payment dates or partial payments this week.

[SMB-0182] Driftwood Partners
  Driftwood Partners has only $297 in the